# 3장 파이썬의 타입 강화: 클린 아키텍처 견고하게 만들기

파이썬으로 구현하는 클린 아키텍처 - 3장 파이썬의 타입 강화: 클린 아키텍처 견고하게 만들기 코드 예제

## 개요

이번 장에서는 파이썬의 강력한 기능인 타입 힌팅type hinting을 다룬다.

이 장에서 다루는 주요 주제:
* 파이썬의 동적 환경에서 타입 인식 이해
* 파이썬 타이핑 시스템 활용
* 정적 타입 검사 자동화 도구 활용

### 00_dynamic_types_vs_type_hinting.py

## 동적 타입 vs 타입 힌팅

파이썬의 동적 타이핑은 빠른 개발을 가능하게 하지만, 런타임 `TypeError`의 원인이 될 수 있다. 타입 힌트를 추가하면 IDE와 정적 분석 도구(mypy)가 실행 전에 오류를 탐지할 수 있다.

> **[의도적 에러]** 아래 셀은 `TypeError`를 시연하는 예제입니다. 정수와 문자열을 더하면 `TypeError: unsupported operand type(s) for +: 'int' and 'str'`가 발생하는 것이 정상입니다.

In [ ]:
# 동적 타입의 유연성과 위험성을 보여주는 예제
x = 5  # x는 정수
x = "hello"  # 이제 x는 문자열이 됨 - 파이썬의 동적 타이핑 특성

"""
이런 유연성 덕분에 빠르게 개발하고 표현력 있는 코드를 작성할 수 있지만,
주의하지 않으면 런타임 오류가 발생할 수 있다. 다음 예제를 살펴보자.
"""


# 타입 힌트 없는 함수 - 어떤 타입이든 받을 수 있어 런타임 오류 가능성 존재
def add_numbers(a, b):
    return a + b


result = add_numbers(5, 3)  # 정상 작동, 결과는 8
result = add_numbers(
    5, "3"
)  # TypeError 발생: + 연산자는 'int'와 'str' 타입을 지원하지 않음


"""
타입 힌트는 파이썬 3.5에서 도입되었으며, 개발자가 변수와 함수 매개변수뿐만 아니라
반환 값에 대해 예상되는 타입을 주석으로 달 수 있게 한다. 타입 힌트와 관련하여,
2장의 add_numbers 함수를 다시 살펴보자.
"""


# 타입 힌트가 추가된 함수 - 매개변수와 반환 타입을 명시하여 의도 전달
# 런타임에는 강제되지 않지만, IDE와 타입 검사기(mypy)가 오류를 사전 탐지
def add_numbers(a: int, b: int) -> int:
    return a + b


result = add_numbers(5, 3)  # 정상 작동, 결과는 8
result = add_numbers(5, "3")  # IDE나 타입 검사기가 오류로 표시함

### 01_shape.py

## Shape 예제에 타입 힌트 적용

타입 힌트를 사용하면 클린 아키텍처의 계층 간 인터페이스가 더 명시적이 된다. 이 예제에서:
- `Shape.area() → float`: 모든 구현체가 준수할 반환 타입 계약
- `AreaCalculator`: 구체 클래스가 아닌 추상 `Shape`에 의존 (의존성 역전)

**참고:** 타입 힌트는 런타임에 강제되지 않으며, 문서화 및 정적 분석 도구용이다.

In [ ]:
# 타입 힌트를 적용한 Shape(도형) 예제
# 추상 클래스와 타입 힌트를 결합하여 계층 간 명확한 인터페이스 정의
import math
from abc import ABC, abstractmethod


# Shape 추상 기본 클래스 - 모든 도형의 인터페이스 (내부 원)
# area() 메서드가 float를 반환한다는 계약 정의
class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        pass


# Rectangle 구체 클래스 - Shape 인터페이스 구현 (외부 원)
# 생성자 매개변수와 반환 타입을 모두 명시하여 사용법 전달
class Rectangle(Shape):
    def __init__(self, width: float, height: float) -> None:
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height


# Circle 구체 클래스 - Shape 인터페이스 구현 (외부 원)
class Circle(Shape):
    def __init__(self, radius: float) -> None:

        self.radius = radius

    def area(self) -> float:
        return math.pi * self.radius**2


# AreaCalculator - Shape 추상 타입에 의존하여 의존성 역전 원칙 준수
# 구체 도형(Rectangle, Circle)이 아닌 추상 Shape에 의존
class AreaCalculator:
    def calculate_area(self, shape: Shape) -> float:

        return shape.area()

### 02_type_hinting_containers.py

## 컨테이너 타입 힌팅 (List, Dict, Set 등)

`list[str]`, `dict[str, int]` 등으로 컬렉션의 요소 타입을 명시한다. 클린 아키텍처에서 계층 간 복잡한 데이터 구조를 주고받을 때 인터페이스를 명확히 하는 데 유용하다.

In [ ]:
# 컨테이너 타입 힌팅 예제 - list, dict 등 컬렉션의 요소 타입 명시
# 계층 간 복잡한 데이터 구조를 주고받을 때 인터페이스를 명확히 하는 기법
def process_order(items: list[str], quantities: list[int]) -> dict[str, int]:
    # items: 문자열 리스트, quantities: 정수 리스트
    # 반환값: 문자열 키-정수 값의 딕셔너리
    return {item: quantity for item, quantity in zip(items, quantities)}


# 사용법
order = process_order(["apple", "banana", "orange"], [2, 3, 1])
print(order)
# 출력: {'apple': 2, 'banana': 3, 'orange': 1}

### 03_type_hinting_sequence.py

## Sequence 타입

`list`나 `tuple` 대신 `Sequence`를 사용하면 SOLID 원칙에 부합하는 유연한 인터페이스를 만들 수 있다:
- **LSP**: 모든 시퀀스 타입과 호환
- **OCP**: 새로운 시퀀스 타입 추가 시 함수 수정 불필요
- **ISP**: 필요한 최소한의 인터페이스(반복)만 요구

In [ ]:
# Sequence 타입 - 리스코프 치환 원칙(LSP)과 개방-폐쇄 원칙(OCP) 준수
# list, tuple 등 특정 타입 대신 Sequence를 사용하여 유연성 확보
from typing import Sequence


# Sequence[float]를 매개변수로 받아 모든 시퀀스 타입(list, tuple 등)과 호환
# → 새로운 시퀀스 타입 추가 시 함수 수정 불필요 (개방-폐쇄 원칙)
def calculate_total(items: Sequence[float]) -> float:
    return sum(items)


# 사용법 - list와 tuple 모두 동작하는 유연한 인터페이스
print(calculate_total([1.0, 2.0, 3.0]))  # 리스트로 동작
print(calculate_total((4.0, 5.0, 6.0)))  # 튜플로도 동작

### 04_type_hinting_union_optional.py

## Union과 Optional 타입

클린 아키텍처에서는 특히 계층 간 경계에서 여러 가능한 타입이나 선택적 값을 처리해야 하는 경우가 많다. 유니온Union 타입과 옵셔널Optional 타입은 이런 상황에 적합하다.

In [ ]:
# Union과 Optional 타입 - 계층 간 경계에서 다양한 타입 처리
from typing import Union, Optional


# Union[str, int] - 문자열 또는 정수를 모두 허용하는 입력 타입
# 외부 계층에서 다양한 형식의 데이터가 들어올 때 유용
def process_input(data: Union[str, int]) -> str:
    return str(data)


# Optional[int] - int 또는 None을 허용 (Union[int, None]의 축약형)
# Optional[str] - 반환값이 str 또는 None일 수 있음을 명시
def find_user(user_id: Optional[int] = None) -> Optional[str]:
    if user_id is None:
        return None
    # ... 사용자 검색 로직 ...
    return "User found"


# 사용법
result1 = process_input("Hello")  # str로 동작
result2 = process_input(42)  # int로 동작
user = find_user()  # 선택적 매개변수

### 05_type_hinting_literals.py

## Literal 타입

리터럴Literal 타입으로 변수가 있는 정확한 값을 지정할 수 있다. 클린 아키텍처로 인터페이스 경계에서 특정 값만 허용하도록 강제할 때 특히 유용하다.

Literal 타입은 더 정밀한 인터페이스를 만들어 유효하지 않은 데이터가 시스템 전체로 퍼지는 것을 방지한다. 이 역할은 계층 간 명확한 경계와 계약을 중시하는 클린 아키텍처의 방향과 잘 맞는다.

In [ ]:
# Literal 타입 - 허용되는 값을 정확히 제한하는 타입 힌트
# 인터페이스 경계에서 유효하지 않은 값의 유입을 방지하는 기법
from typing import Literal

# 로그 레벨로 허용되는 정확한 문자열 값을 제한
LogLevel = Literal["DEBUG", "INFO", "WARNING", "ERROR"]


# LogLevel에 정의된 4개의 값만 허용 - 잘못된 값 전달 시 타입 검사기가 경고
def set_log_level(level: LogLevel) -> None:
    print(f"Setting log level to {level}")


# 사용법
set_log_level("DEBUG")  # 유효함
set_log_level("CRITICAL")  # 타입 검사기가 오류로 표시

### 타입 별칭 (Type Aliases)

## 복잡한 타입 주석 단순화

타입 별칭은 복잡한 타입에 읽기 쉬운 이름을 붙여 코드 명확성을 높인다. 새로운 타입을 만들지 않으면서도 **DDD의 데이터 전송 객체(DTO)**를 다룰 때 유용하다. 관심사 분리, 유지 보수성, 명확성을 높이는 데 도움을 준다.

In [ ]:
# 타입 별칭(Type Aliases) - 복잡한 타입에 읽기 쉬운 이름 부여
# DDD의 DTO(데이터 전송 객체)를 다룰 때 코드 명확성을 높이는 기법
UserDict = dict[str, str]       # 개별 사용자 정보를 담는 딕셔너리 타입 별칭
UserList = list[UserDict]       # 사용자 목록을 담는 리스트 타입 별칭

# 타입 별칭 사용으로 매개변수의 의미가 즉시 전달
def process_users(users: UserList) -> None:
    for user in users:
        print(f"Processing user: {user['name']}")

# 사용법
users: UserList = [{"name": "Alice"}, {"name": "Bob"}]
process_users(users)

### 06_type_hinting_new_type.py

## NewType

`NewType`은 정적 타입 검사기가 인식하는 별도의 타입을 생성하여, 기본 타입은 같지만 개념적으로 다른 값(예: `UserId`와 `ProductId`)의 혼용을 방지한다. 타입 별칭보다 강력한 도메인 안전장치이다.

In [ ]:
# NewType - 타입 별칭보다 강력한 별개의 타입 생성
# 기본 타입은 같지만 개념적으로 다른 값의 혼용을 방지하는 도메인 안전장치
from typing import NewType

# UserId와 ProductId는 모두 int 기반이지만, 서로 다른 도메인 개념
UserId = NewType("UserId", int)       # 사용자 식별자 전용 타입
ProductId = NewType("ProductId", int)  # 상품 식별자 전용 타입


# 매개변수 순서를 잘못 전달하면 타입 검사기가 오류를 탐지
def process_order(user_id: UserId, product_id: ProductId) -> None:
    print(f"Processing order for User {user_id} and Product {product_id}")


# 사용법
user_id = UserId(1)
product_id = ProductId(1)  # 기본 타입은 같은 int지만 별개의 타입
process_order(user_id, product_id)
# 아래 코드는 타입 오류가 발생함:
# process_order(product_id, user_id)

### 07_type_hinting_any_type.py

## Any 타입

`Any`는 모든 타입을 허용하는 특별한 타입 힌트다. 클린 아키텍처에서는 계층 경계의 타입을 가능한 한 구체적으로 지정하는 것이 원칙이므로, `Any`는 **최후의 수단**으로만 사용해야 한다. 자신의 코드에서 `Any`를 사용하고 있다면 구체적 타입으로 대체할 수 있는지 검토하는 것이 좋다.

In [ ]:
# Any 타입 - 모든 타입을 허용하는 특별한 타입 힌트
# 클린 아키텍처에서는 최후의 수단으로만 사용 권장 (구체적 타입 지정이 원칙)
from typing import Any


# Any를 사용하면 타입 검사기가 타입 관련 오류를 탐지하지 않음
# → 외부 시스템 연동 등 타입을 알 수 없는 경우에만 제한적 사용 권장
def log_data(data: Any) -> None:
    print(f"Logged: {data}")


# 사용법 - 문자열, 정수, 딕셔너리 등 어떤 타입이든 전달 가능
log_data("문자열")
log_data(42)
log_data({"key": "value"})

### 08_type_hinting_mypy_cli.py

## mypy CLI 사용법

설치가 완료되면 mypy를 사용하여 파이썬 파일의 타입 오류를 확인할 수 있다. 간단한 예시를 ﻿살펴보자.

In [ ]:
# mypy CLI 사용 예제 - 정적 타입 검사 도구로 런타임 전 오류 탐지
# 클린 아키텍처에서 계층 간 인터페이스의 타입 안전성을 자동으로 검증하는 도구

# 반환 타입이 dict로 명시된 사용자 조회 함수
def get_user(user_id: int) -> dict:
    # 사용자 조회 시뮬레이션
    return {"id": user_id, "name": "John Doe", "email": "john@example.com"}


# dict 타입의 user와 str 타입의 subject를 받는 이메일 발송 함수
def send_email(user: dict, subject: str) -> None:
    print(f"Sending email to {user['email']} with subject: {subject}")


# 사용법 - "123"은 str이지만 user_id는 int를 기대 → mypy가 타입 불일치 탐지
user = get_user("123")
send_email(user, "Welcome!")

"""
mypy Chapter_3/08_type_hinting_mypy_cli.py
Chapter_3/08_type_hinting_mypy_cli.py:11: error: Argument 1 to "get_user" has incompatible type "str"; expected "int"  [arg-type]
Found 1 error in 1 file (checked 1 source file)
"""